# Neural Receiver - 전체 시뮬레이션
## 약한 신호 검출 및 파라미터 분류를 위한 End-to-End 시뮬레이션

이 노트북은 다음을 수행합니다:
1. ✅ 환경 설정 및 데이터 생성
2. ✅ 멀티태스크 신경망 수신기 학습
3. ✅ 성능 평가 및 시각화
4. ✅ 다양한 SNR에서의 검출/분류 성능 분석

## Step 1: 환경 설정

In [ ]:
# 저장소 클론 (Colab에서 실행 시)
import os
import sys

# Colab 환경 확인
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo_name = "Neural-Receiver-for-Weak-Signal-Detection-and-Parametric-Classification"
    if not os.path.exists(repo_name):
        !git clone https://github.com/hyeonhwilee/Neural-Receiver-for-Weak-Signal-Detection-and-Parametric-Classification.git
        %cd {repo_name}
    else:
        %cd {repo_name}
    
    # 패키지 설치
    !pip install -q torch torchvision torchaudio
    !pip install -q numpy scipy matplotlib seaborn scikit-learn pandas tqdm tensorboard

print("✓ 환경 설정 완료")

In [ ]:
# 라이브러리 임포트
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

# 시각화 설정
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n📱 Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# 랜덤 시드 고정
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    
set_seed(42)
print("\n✓ 라이브러리 임포트 및 시드 설정 완료")

## Step 2: 데이터 생성 및 데이터셋 정의

In [ ]:
class IQSignalGenerator:
    """
    다양한 변조 방식과 SNR에서 IQ 신호를 생성하는 클래스
    """
    def __init__(self, sample_rate=1000):
        self.sample_rate = sample_rate
        self.modulation_types = ['BPSK', 'QPSK', '8PSK', 'QAM16']
        
    def generate_signal(self, num_symbols=100, modulation='BPSK', snr_db=0, add_signal=True):
        """
        IQ 신호 생성
        
        Args:
            num_symbols: 심볼 수
            modulation: 변조 방식 (BPSK, QPSK, 8PSK, QAM16)
            snr_db: Signal-to-Noise Ratio (dB)
            add_signal: True면 신호+노이즈, False면 노이즈만
        """
        samples_per_symbol = 10
        total_samples = num_symbols * samples_per_symbol
        
        if add_signal:
            # 변조 신호 생성
            if modulation == 'BPSK':
                symbols = np.random.choice([-1, 1], num_symbols)
                constellation = symbols + 0j
            elif modulation == 'QPSK':
                symbols = np.random.choice([1+1j, 1-1j, -1+1j, -1-1j], num_symbols) / np.sqrt(2)
                constellation = symbols
            elif modulation == '8PSK':
                phases = np.random.choice(np.arange(8), num_symbols) * 2 * np.pi / 8
                constellation = np.exp(1j * phases)
            elif modulation == 'QAM16':
                real_part = np.random.choice([-3, -1, 1, 3], num_symbols)
                imag_part = np.random.choice([-3, -1, 1, 3], num_symbols)
                constellation = (real_part + 1j * imag_part) / np.sqrt(10)
            
            # 업샘플링 (펄스 성형)
            signal = np.repeat(constellation, samples_per_symbol)
            
            # SNR 적용
            signal_power = np.mean(np.abs(signal)**2)
            snr_linear = 10 ** (snr_db / 10)
            noise_power = signal_power / snr_linear
        else:
            signal = np.zeros(total_samples, dtype=complex)
            noise_power = 1.0
        
        # 가우시안 복소 노이즈 추가
        noise = np.sqrt(noise_power / 2) * (np.random.randn(total_samples) + 1j * np.random.randn(total_samples))
        received_signal = signal + noise
        
        return received_signal
    
    def get_modulation_index(self, modulation):
        return self.modulation_types.index(modulation)


class IQDataset(Dataset):
    """
    PyTorch Dataset for IQ signals
    """
    def __init__(self, num_samples=1000, snr_range=(-20, 20), signal_ratio=0.7):
        self.generator = IQSignalGenerator()
        self.num_samples = num_samples
        self.snr_range = snr_range
        self.signal_ratio = signal_ratio
        
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        # 신호 유무 결정
        has_signal = np.random.rand() < self.signal_ratio
        
        # SNR 랜덤 선택
        snr_db = np.random.uniform(self.snr_range[0], self.snr_range[1])
        
        if has_signal:
            # 변조 방식 랜덤 선택
            modulation = np.random.choice(self.generator.modulation_types)
            signal = self.generator.generate_signal(modulation=modulation, snr_db=snr_db, add_signal=True)
            label = self.generator.get_modulation_index(modulation)
        else:
            # 노이즈만
            signal = self.generator.generate_signal(add_signal=False)
            label = 0  # 노이즈일 때는 레이블이 의미 없지만 0으로 설정
        
        # I/Q 성분 추출 및 정규화
        i_component = signal.real
        q_component = signal.imag
        
        # 특징 추출: 통계량 계산
        features = np.array([
            np.mean(i_component), np.std(i_component),
            np.mean(q_component), np.std(q_component),
            np.mean(np.abs(signal)), np.std(np.abs(signal)),
            np.mean(np.angle(signal)), np.std(np.angle(signal)),
        ], dtype=np.float32)
        
        detection_label = float(has_signal)
        
        return torch.FloatTensor(features), torch.FloatTensor([detection_label]), torch.LongTensor([label])

# 데이터셋 생성
train_dataset = IQDataset(num_samples=10000, snr_range=(-20, 20), signal_ratio=0.7)
val_dataset = IQDataset(num_samples=2000, snr_range=(-20, 20), signal_ratio=0.7)
test_dataset = IQDataset(num_samples=2000, snr_range=(-20, 20), signal_ratio=0.7)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f"\n✓ 데이터셋 생성 완료")
print(f"  - Train: {len(train_dataset)} samples")
print(f"  - Val: {len(val_dataset)} samples")
print(f"  - Test: {len(test_dataset)} samples")
print(f"  - Feature dimension: 8")
print(f"  - Modulation types: {train_dataset.generator.modulation_types}")

In [ ]:
# 샘플 데이터 시각화
generator = IQSignalGenerator()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('다양한 변조 방식의 IQ 신호 (SNR = 10 dB)', fontsize=14, fontweight='bold')

for idx, modulation in enumerate(generator.modulation_types):
    signal = generator.generate_signal(num_symbols=100, modulation=modulation, snr_db=10)
    
    # I/Q Time Series
    axes[0, idx].plot(signal.real[:200], label='I', alpha=0.7)
    axes[0, idx].plot(signal.imag[:200], label='Q', alpha=0.7)
    axes[0, idx].set_title(f'{modulation}\nTime Series')
    axes[0, idx].set_xlabel('Sample')
    axes[0, idx].set_ylabel('Amplitude')
    axes[0, idx].legend()
    axes[0, idx].grid(True, alpha=0.3)
    
    # Constellation
    axes[1, idx].scatter(signal.real, signal.imag, alpha=0.2, s=2)
    axes[1, idx].set_title(f'{modulation}\nConstellation')
    axes[1, idx].set_xlabel('I')
    axes[1, idx].set_ylabel('Q')
    axes[1, idx].grid(True, alpha=0.3)
    axes[1, idx].axis('equal')

plt.tight_layout()
plt.show()

## Step 3: 멀티태스크 신경망 수신기 정의

In [ ]:
class MultiTaskNeuralReceiver(nn.Module):
    """
    멀티태스크 신경망 수신기
    - Task 1: 신호 검출 (이진 분류)
    - Task 2: 변조 방식 분류 (다중 분류)
    """
    def __init__(self, input_dim=8, hidden_dim=128, num_modulations=4, dropout=0.3):
        super(MultiTaskNeuralReceiver, self).__init__()
        
        # Shared feature extractor
        self.shared_layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        
        # Task 1: Signal Detection Head
        self.detection_head = nn.Sequential(
            nn.Linear(hidden_dim // 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
        # Task 2: Modulation Classification Head
        self.classification_head = nn.Sequential(
            nn.Linear(hidden_dim // 2, 64),
            nn.ReLU(),
            nn.Linear(64, num_modulations)
        )
        
    def forward(self, x):
        # Shared features
        features = self.shared_layers(x)
        
        # Detection output
        detection = self.detection_head(features)
        
        # Classification output
        classification = self.classification_head(features)
        
        return detection, classification

# 모델 생성
model = MultiTaskNeuralReceiver(
    input_dim=8,
    hidden_dim=128,
    num_modulations=4,
    dropout=0.3
).to(device)

# 모델 정보 출력
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✓ 모델 생성 완료")
print(f"  - Total parameters: {total_params:,}")
print(f"  - Trainable parameters: {trainable_params:,}")
print(f"\n{model}")

## Step 4: 학습 설정 및 실행

In [ ]:
# 손실 함수 및 옵티마이저
criterion_detection = nn.BCELoss()
criterion_classification = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# 학습 함수
def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    detection_loss_sum = 0
    classification_loss_sum = 0
    
    for features, detection_labels, classification_labels in tqdm(dataloader, desc="Training", leave=False):
        features = features.to(device)
        detection_labels = detection_labels.to(device)
        classification_labels = classification_labels.squeeze().to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        detection_pred, classification_pred = model(features)
        
        # 손실 계산
        loss_det = criterion_detection(detection_pred, detection_labels)
        loss_cls = criterion_classification(classification_pred, classification_labels)
        
        # 멀티태스크 손실 (가중치 조정 가능)
        loss = loss_det + 0.5 * loss_cls
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        detection_loss_sum += loss_det.item()
        classification_loss_sum += loss_cls.item()
    
    return total_loss / len(dataloader), detection_loss_sum / len(dataloader), classification_loss_sum / len(dataloader)

def validate(model, dataloader, device):
    model.eval()
    total_loss = 0
    detection_loss_sum = 0
    classification_loss_sum = 0
    
    with torch.no_grad():
        for features, detection_labels, classification_labels in tqdm(dataloader, desc="Validation", leave=False):
            features = features.to(device)
            detection_labels = detection_labels.to(device)
            classification_labels = classification_labels.squeeze().to(device)
            
            detection_pred, classification_pred = model(features)
            
            loss_det = criterion_detection(detection_pred, detection_labels)
            loss_cls = criterion_classification(classification_pred, classification_labels)
            loss = loss_det + 0.5 * loss_cls
            
            total_loss += loss.item()
            detection_loss_sum += loss_det.item()
            classification_loss_sum += loss_cls.item()
    
    return total_loss / len(dataloader), detection_loss_sum / len(dataloader), classification_loss_sum / len(dataloader)

print("\n✓ 학습 설정 완료")
print(f"  - Optimizer: Adam (lr=0.001)")
print(f"  - Scheduler: ReduceLROnPlateau")
print(f"  - Detection Loss: Binary Cross Entropy")
print(f"  - Classification Loss: Cross Entropy")

In [ ]:
# 학습 실행
num_epochs = 30
history = {
    'train_loss': [], 'val_loss': [],
    'train_det_loss': [], 'val_det_loss': [],
    'train_cls_loss': [], 'val_cls_loss': []
}

best_val_loss = float('inf')

print("\n🚀 학습 시작...\n")

for epoch in range(num_epochs):
    # 학습
    train_loss, train_det_loss, train_cls_loss = train_epoch(model, train_loader, optimizer, device)
    
    # 검증
    val_loss, val_det_loss, val_cls_loss = validate(model, val_loader, device)
    
    # 스케줄러 업데이트
    scheduler.step(val_loss)
    
    # 기록
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_det_loss'].append(train_det_loss)
    history['val_det_loss'].append(val_det_loss)
    history['train_cls_loss'].append(train_cls_loss)
    history['val_cls_loss'].append(val_cls_loss)
    
    # 최고 모델 저장
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        best_epoch = epoch
    
    # 진행 상황 출력
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train Loss: {train_loss:.4f} (Det: {train_det_loss:.4f}, Cls: {train_cls_loss:.4f})")
        print(f"  Val Loss:   {val_loss:.4f} (Det: {val_det_loss:.4f}, Cls: {val_cls_loss:.4f})")
        print(f"  Best Val Loss: {best_val_loss:.4f} (Epoch {best_epoch+1})\n")

print("\n✓ 학습 완료!")
print(f"  Best validation loss: {best_val_loss:.4f} at epoch {best_epoch+1}")

In [ ]:
# 학습 곡선 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Total Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0].axvline(best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best (Epoch {best_epoch+1})')
axes[0].set_title('Total Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Detection Loss
axes[1].plot(history['train_det_loss'], label='Train', linewidth=2)
axes[1].plot(history['val_det_loss'], label='Validation', linewidth=2)
axes[1].set_title('Detection Loss (BCE)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Classification Loss
axes[2].plot(history['train_cls_loss'], label='Train', linewidth=2)
axes[2].plot(history['val_cls_loss'], label='Validation', linewidth=2)
axes[2].set_title('Classification Loss (CE)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: 테스트 및 성능 평가

In [ ]:
# 최고 모델 로드
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# 테스트 데이터로 예측
all_detection_preds = []
all_detection_labels = []
all_classification_preds = []
all_classification_labels = []

with torch.no_grad():
    for features, detection_labels, classification_labels in tqdm(test_loader, desc="Testing"):
        features = features.to(device)
        
        detection_pred, classification_pred = model(features)
        
        all_detection_preds.extend(detection_pred.cpu().numpy())
        all_detection_labels.extend(detection_labels.numpy())
        all_classification_preds.extend(classification_pred.argmax(dim=1).cpu().numpy())
        all_classification_labels.extend(classification_labels.squeeze().numpy())

all_detection_preds = np.array(all_detection_preds).flatten()
all_detection_labels = np.array(all_detection_labels).flatten()
all_classification_preds = np.array(all_classification_preds)
all_classification_labels = np.array(all_classification_labels)

print("\n✓ 테스트 완료")

In [ ]:
# Detection 성능 평가
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

detection_binary = (all_detection_preds > 0.5).astype(int)

det_accuracy = accuracy_score(all_detection_labels, detection_binary)
det_precision = precision_score(all_detection_labels, detection_binary)
det_recall = recall_score(all_detection_labels, detection_binary)
det_f1 = f1_score(all_detection_labels, detection_binary)

print("\n" + "="*60)
print("📊 신호 검출 성능 (Signal Detection Performance)")
print("="*60)
print(f"Accuracy:  {det_accuracy:.4f} ({det_accuracy*100:.2f}%)")
print(f"Precision: {det_precision:.4f}")
print(f"Recall:    {det_recall:.4f}")
print(f"F1-Score:  {det_f1:.4f}")
print("="*60)

# ROC 곡선
fpr, tpr, thresholds = roc_curve(all_detection_labels, all_detection_preds)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, linewidth=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve - Signal Detection', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Detection Score Distribution
axes[1].hist(all_detection_preds[all_detection_labels == 0], bins=50, alpha=0.5, label='No Signal', density=True)
axes[1].hist(all_detection_preds[all_detection_labels == 1], bins=50, alpha=0.5, label='Signal', density=True)
axes[1].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Threshold')
axes[1].set_xlabel('Detection Score')
axes[1].set_ylabel('Density')
axes[1].set_title('Detection Score Distribution', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Classification 성능 평가 (신호가 있는 경우만)
signal_idx = all_detection_labels == 1
cls_preds = all_classification_preds[signal_idx]
cls_labels = all_classification_labels[signal_idx]

cls_accuracy = accuracy_score(cls_labels, cls_preds)

print("\n" + "="*60)
print("📊 변조 방식 분류 성능 (Modulation Classification)")
print("="*60)
print(f"Accuracy: {cls_accuracy:.4f} ({cls_accuracy*100:.2f}%)")
print("="*60)
print("\nClassification Report:")
print(classification_report(cls_labels, cls_preds, 
                          target_names=train_dataset.generator.modulation_types))

# Confusion Matrix
cm = confusion_matrix(cls_labels, cls_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=train_dataset.generator.modulation_types,
            yticklabels=train_dataset.generator.modulation_types)
plt.title('Confusion Matrix - Modulation Classification', fontweight='bold', fontsize=12)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## Step 6: SNR에 따른 성능 분석

In [ ]:
# 다양한 SNR에서 성능 평가
snr_values = np.arange(-20, 21, 5)
detection_accuracies = []
classification_accuracies = []

print("\n다양한 SNR에서 성능 평가 중...\n")

for snr in tqdm(snr_values):
    # 특정 SNR로 테스트 데이터셋 생성
    test_dataset_snr = IQDataset(num_samples=1000, snr_range=(snr, snr), signal_ratio=0.7)
    test_loader_snr = DataLoader(test_dataset_snr, batch_size=64, shuffle=False)
    
    det_preds = []
    det_labels = []
    cls_preds = []
    cls_labels = []
    
    with torch.no_grad():
        for features, detection_label, classification_label in test_loader_snr:
            features = features.to(device)
            detection_pred, classification_pred = model(features)
            
            det_preds.extend(detection_pred.cpu().numpy())
            det_labels.extend(detection_label.numpy())
            cls_preds.extend(classification_pred.argmax(dim=1).cpu().numpy())
            cls_labels.extend(classification_label.squeeze().numpy())
    
    det_preds = np.array(det_preds).flatten()
    det_labels = np.array(det_labels).flatten()
    cls_preds = np.array(cls_preds)
    cls_labels = np.array(cls_labels)
    
    # Detection accuracy
    det_binary = (det_preds > 0.5).astype(int)
    det_acc = accuracy_score(det_labels, det_binary)
    detection_accuracies.append(det_acc)
    
    # Classification accuracy (신호가 있는 경우만)
    signal_idx = det_labels == 1
    if signal_idx.sum() > 0:
        cls_acc = accuracy_score(cls_labels[signal_idx], cls_preds[signal_idx])
        classification_accuracies.append(cls_acc)
    else:
        classification_accuracies.append(0)

# SNR vs Accuracy 그래프
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(snr_values, np.array(detection_accuracies) * 100, marker='o', linewidth=2, 
        markersize=8, label='Signal Detection', color='#2E86AB')
ax.plot(snr_values, np.array(classification_accuracies) * 100, marker='s', linewidth=2, 
        markersize=8, label='Modulation Classification', color='#A23B72')

ax.axhline(90, color='green', linestyle='--', alpha=0.5, linewidth=1, label='90% Accuracy')
ax.axhline(95, color='red', linestyle='--', alpha=0.5, linewidth=1, label='95% Accuracy')

ax.set_xlabel('SNR (dB)', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Performance vs SNR', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
ax.set_ylim([0, 105])

plt.tight_layout()
plt.show()

# 결과 테이블
results_df = pd.DataFrame({
    'SNR (dB)': snr_values,
    'Detection Acc (%)': np.array(detection_accuracies) * 100,
    'Classification Acc (%)': np.array(classification_accuracies) * 100
})

print("\n" + "="*60)
print("📊 SNR별 성능 요약")
print("="*60)
print(results_df.to_string(index=False))
print("="*60)

## Step 7: 최종 요약

In [ ]:
print("\n" + "="*80)
print("🎉 시뮬레이션 완료!")
print("="*80)
print(f"\n✅ 모델 학습 완료")
print(f"   - Best Validation Loss: {best_val_loss:.4f}")
print(f"   - Best Epoch: {best_epoch + 1}/{num_epochs}")
print(f"\n✅ 테스트 성능")
print(f"   - Signal Detection Accuracy: {det_accuracy*100:.2f}%")
print(f"   - Signal Detection F1-Score: {det_f1:.4f}")
print(f"   - ROC-AUC: {roc_auc:.4f}")
print(f"   - Modulation Classification Accuracy: {cls_accuracy*100:.2f}%")
print(f"\n✅ SNR 성능 범위")
print(f"   - Tested SNR: {snr_values.min()} dB to {snr_values.max()} dB")
print(f"   - Best Detection Accuracy: {max(detection_accuracies)*100:.2f}% at {snr_values[np.argmax(detection_accuracies)]} dB")
print(f"   - Best Classification Accuracy: {max(classification_accuracies)*100:.2f}% at {snr_values[np.argmax(classification_accuracies)]} dB")
print(f"\n✅ 저장된 파일")
print(f"   - best_model.pth (최고 성능 모델)")
print("\n" + "="*80)
print("\n💡 이 모델은 약한 신호 환경(-20dB ~ 20dB SNR)에서")
print("   신호 검출과 변조 방식 분류를 동시에 수행할 수 있습니다!")
print("="*80)